In [0]:
spark

In [0]:
myRange = spark.range(1000).toDF("number")

In [0]:
myRange.show()

+------+
|number|
+------+
|     0|
|     1|
|     2|
|     3|
|     4|
|     5|
|     6|
|     7|
|     8|
|     9|
|    10|
|    11|
|    12|
|    13|
|    14|
|    15|
|    16|
|    17|
|    18|
|    19|
+------+
only showing top 20 rows


In [0]:
divisBy2 = myRange.where("number % 2 = 0")
divisBy2.count()

500

In [0]:
flightData2015 = spark.read.csv("/Volumes/workspace/default/raw_layer/2015-summary.csv", header= True, inferSchema=True)
flightData2015.show()
# /Workspace/Users/uad.fiverr@gmail.com/pyspark_fundamentals/data/flight_data/2015-summary.csv

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|   15|
|       United States|            Croatia|    1|
|       United States|            Ireland|  344|
|               Egypt|      United States|   15|
|       United States|              India|   62|
|       United States|          Singapore|    1|
|       United States|            Grenada|   62|
|          Costa Rica|      United States|  588|
|             Senegal|      United States|   40|
|             Moldova|      United States|    1|
|       United States|       Sint Maarten|  325|
|       United States|   Marshall Islands|   39|
|              Guyana|      United States|   64|
|               Malta|      United States|    1|
|            Anguilla|      United States|   41|
|             Bolivia|      United States|   30|
|       United States|           Paraguay|    6|
|             Algeri

In [0]:
flightData2015.take(5)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count=344),
 Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count=62)]

In [0]:
flightData2015.sort("count").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonSort [count#13353 ASC NULLS FIRST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#8547]
               +- PhotonShuffleExchangeSink rangepartitioning(count#13353 ASC NULLS FIRST, 5)
                  +- PhotonRowToColumnar
                     +- FileScan csv [DEST_COUNTRY_NAME#13351,ORIGIN_COUNTRY_NAME#13352,count#13353] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/raw_layer/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "5")
flightData2015.sort("count").take(2)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', count=1)]

In [0]:
flightData2015.createOrReplaceTempView("flight_data_2015")
sqlWay = spark.sql("""SELECT DEST_COUNTRY_NAME, COUNT(1) FROM flight_data_2015 GROUP BY DEST_COUNTRY_NAME""")

In [0]:
dataFrameWay = flightData2015\
.groupBy("DEST_COUNTRY_NAME")\
.count()

In [0]:
sqlWay.explain()
dataFrameWay.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#13351], functions=[finalmerge_count(merge count#13393L) AS count(1)#13390L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#8618]
               +- PhotonShuffleExchangeSink hashpartitioning(DEST_COUNTRY_NAME#13351, 5)
                  +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#13351], functions=[partial_count(1) AS count#13393L])
                     +- PhotonRowToColumnar
                        +- FileScan csv [DEST_COUNTRY_NAME#13351] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/raw_layer/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>


== Photon Explanation ==
The query is fully supported by Photon.
== Physical Plan ==
AdaptiveSpa

In [0]:
spark.sql("SELECT max(count) FROM flight_data_2015").take(1)

[Row(max(count)=370002)]

In [0]:
from pyspark.sql.functions import max
flightData2015.select(max("count")).take(1)

[Row(max(count)=370002)]

In [0]:
maxSql = spark.sql("""SELECT DEST_COUNTRY_NAME, SUM(count) as DESTINATION_TOTAL
                   FROM flight_data_2015 
                   GROUP BY DEST_COUNTRY_NAME 
                   ORDER BY SUM(count) DESC
                   LIMIT 5""")
maxSql.show()

+-----------------+-----------------+
|DEST_COUNTRY_NAME|DESTINATION_TOTAL|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+



In [0]:
flightData2015.groupBy("DEST_COUNTRY_NAME").sum("count").withColumnRenamed("sum(count)", "DESTINATION_TOTAL").sort("DESTINATION_TOTAL", ascending=False).limit(5).show()

+-----------------+-----------------+
|DEST_COUNTRY_NAME|DESTINATION_TOTAL|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+

